# Battle Lab · Benchmark HOLDOUT final M-C 🐉⚔️

Examen final de `BATTLE-LAB-MC-TRAIN-001`.

- Usa **únicamente los 36 equipos holdout** (`signatureLeakage = 0`).
- Compara el **BC público M-A/M-B** contra el **LIGHT M-C 196,608**.
- Ambos checkpoints reciben la **misma agenda determinista y espejada**.
- Cada modelo juega 500 combates contra Random, 500 contra Max Base Power y 500 contra Simple Heuristics: **3,000 batallas totales**.
- Guarda resultados por bloques de 500, así que una desconexión puede reutilizar controles ya concluidos.
- `SimpleHeuristicsPlayer` es la métrica primaria; Random y Max Base Power son controles de regresión.


In [ ]:
BATTLES_PER_CONTROL = 500  # @param {type:"integer"}
DEVICE = "auto"  # @param ["auto", "cuda", "cpu"]
SEED = 260913
PORT = 8000
RESUME = True
PKMN_REPOSITORY = "https://github.com/Iesyo/pkmn.git"
PKMN_REF = "main"
FINAL_SHA256 = "fa8687d08feeb169f4eb4f4a078b65971346e2ef5b0ca0ff899e811721075759"
NODE_VERSION = "24.21.0"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/Colabs/LikeNoOneEverWas/BattleLab/MC-Training")
SPLIT_ROOT = DRIVE_ROOT / "data" / "team-splits" / f"seed-{SEED}"
HOLDOUT_DIR = SPLIT_ROOT / "holdout"
SPLIT_MANIFEST = SPLIT_ROOT / "split_manifest.json"
BASELINE_CHECKPOINT = DRIVE_ROOT / "training" / "baseline" / "vgc-bench-ma-mb-100.zip"
LIGHT_CHECKPOINT = DRIVE_ROOT / "training" / "rl" / "light" / f"seed{SEED}" / "checkpoints" / "step-000196608.zip"
OUTPUT_DIR = DRIVE_ROOT / "benchmark-holdout"
RUNTIME_ROOT = Path("/content/battle-lab-runtime/holdout-final")
PKMN_ROOT = Path("/content/pkmn")
for p in (OUTPUT_DIR, RUNTIME_ROOT.parent): p.mkdir(parents=True, exist_ok=True)
print("📁 Holdout:", HOLDOUT_DIR)
print("📁 Resultados:", OUTPUT_DIR)


In [ ]:
import json, os, subprocess, sys

def run(command, *, cwd=None, label=None):
    if label: print(f"\n▶ {label}", flush=True)
    subprocess.run([str(x) for x in command], cwd=cwd, check=True)

if not (PKMN_ROOT / ".git").is_dir():
    run(["git", "clone", "--filter=blob:none", PKMN_REPOSITORY, str(PKMN_ROOT)], label="Clonando pkmn")
run(["git", "fetch", "origin", PKMN_REF], cwd=PKMN_ROOT, label="Actualizando pkmn")
run(["git", "checkout", "--force", "FETCH_HEAD"], cwd=PKMN_ROOT, label="Fijando pkmn")
pkmn_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PKMN_ROOT, text=True).strip()
print("pkmn SHA:", pkmn_sha)

node_major = int(subprocess.check_output(["node", "-p", "process.versions.node.split('.')[0]"], text=True).strip())
if node_major < 24:
    run(["npm", "install", "-g", "n"], label="Instalando selector Node")
    run(["n", NODE_VERSION], label=f"Node {NODE_VERSION}")
    os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PKMN_ROOT / "battle_lab" / "requirements-phase1.txt")], label="Dependencias Battle Lab")
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PKMN_ROOT / "battle_lab" / "requirements-phase2.txt")], label="Dependencias benchmark")

split = json.loads(SPLIT_MANIFEST.read_text())
if int(split.get("holdoutTeams", -1)) != 36 or int(split.get("signatureLeakage", -1)) != 0:
    raise RuntimeError(f"Split holdout inválido: {split}")
if not LIGHT_CHECKPOINT.is_file():
    raise RuntimeError(f"Falta checkpoint LIGHT: {LIGHT_CHECKPOINT}")
print("✅ Holdout verificado: 36 equipos · fuga=0")
print("⏱️ Referencia: 3,000 batallas; la corrida calibrada de 1,500 tomó ~25 min. La ETA real aparecerá durante la ejecución.")


In [ ]:
cmd = [
    sys.executable, "-u", "-m", "battle_lab.mc_holdout_benchmark",
    "--runtime-root", str(RUNTIME_ROOT),
    "--output-dir", str(OUTPUT_DIR),
    "--holdout-dir", str(HOLDOUT_DIR),
    "--split-manifest", str(SPLIT_MANIFEST),
    "--baseline-checkpoint", str(BASELINE_CHECKPOINT),
    "--candidate-checkpoint", str(LIGHT_CHECKPOINT),
    "--candidate-sha256", FINAL_SHA256,
    "--benchmark-battles-per-baseline", str(BATTLES_PER_CONTROL),
    "--seed", str(SEED),
    "--port", str(PORT),
    "--device", DEVICE,
]
cmd.append("--resume" if RESUME else "--no-resume")
print("🚦 Iniciando benchmark HOLDOUT. Debe mostrar barra/ETA durante cada bloque de 500.", flush=True)
run(cmd, cwd=PKMN_ROOT, label="BC público vs LIGHT · 36 holdout · 3 controles")


In [ ]:
result_path = OUTPUT_DIR / "latest_result.json"
status_path = OUTPUT_DIR / "status.json"
if not result_path.is_file():
    status = json.loads(status_path.read_text()) if status_path.is_file() else {}
    raise RuntimeError(f"El benchmark no terminó. Estado: {status}")
result = json.loads(result_path.read_text())
cmp = result["comparison"]
print("\n===== BATTLE LAB · HOLDOUT FINAL =====")
print("Estado:", result["state"])
print("Batallas:", result["protocol"]["totalBattles"])
print("Agenda SHA:", result["protocol"]["schedule"]["sha256"])
print("BC público global:", f"{cmp['publicOverallScorePercent']:.2f}%")
print("LIGHT global:", f"{cmp['lightOverallScorePercent']:.2f}%")
print("Δ global:", f"{cmp['overallDeltaPercentagePoints']:+.2f} pp")
print("\nPor control:")
for control_id, row in cmp["controls"].items():
    print(f"- {row['label']}: {row['publicScorePercent']:.2f}% → {row['lightScorePercent']:.2f}% ({row['deltaPercentagePoints']:+.2f} pp)")
print("\nCambios en celdas de agenda:", cmp["scheduleMatched"])
print("VEREDICTO:", cmp["verdict"].upper())
print("Resultado:", result_path)
print("Replays:", result["artifacts"]["replaysZip"])
